# Data Collection — Web Scraping

**IBM Data Science Capstone — SpaceX Falcon 9**

Repository: [https://github.com/MCerros/IBM-Data-Science-Capstone-SpaceX](https://github.com/MCerros/IBM-Data-Science-Capstone-SpaceX)

Direct notebook URL after upload:  
[https://github.com/MCerros/IBM-Data-Science-Capstone-SpaceX/blob/main/02_SpaceX_Web_Scraping.ipynb](https://github.com/MCerros/IBM-Data-Science-Capstone-SpaceX/blob/main/02_SpaceX_Web_Scraping.ipynb)

**Data integrity note:** This notebook uses the project CSV files stored in the same repository.
No rows, metrics, charts, or model scores are manually invented.

## Objective

Collect Falcon 9 / Falcon Heavy launch records from the fixed Wikipedia snapshot
used by the capstone, parse the launch tables with BeautifulSoup, and store the
result in a Pandas DataFrame.

In [1]:
from pathlib import Path
import pandas as pd
import requests
from bs4 import BeautifulSoup
import unicodedata

WIKI_URL = (
    "https://en.wikipedia.org/w/index.php?"
    "title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"
)
SAVED_FILE = Path("spacex_web_scraped.csv")
print("Scraping source:", WIKI_URL)

Scraping source: https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922


In [2]:
def date_time(table_cells):
    return [x.strip() for x in list(table_cells.strings)][0:2]

def booster_version(table_cells):
    return "".join(
        [x for i, x in enumerate(table_cells.strings) if i % 2 == 0][0:-1]
    )

def landing_status(table_cells):
    return [x for x in table_cells.strings][0]

def get_mass(table_cells):
    mass = unicodedata.normalize("NFKD", table_cells.text).strip()
    if mass:
        return mass[0:mass.find("kg") + 2]
    return 0

def scrape_launch_table(url):
    response = requests.get(
        url,
        headers={"User-Agent": "Mozilla/5.0 IBM-Capstone-Project"},
        timeout=15,
    )
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")

    launch_dict = {
        "Flight No.": [], "Launch site": [], "Payload": [], "Payload mass": [],
        "Orbit": [], "Customer": [], "Launch outcome": [], "Version Booster": [],
        "Booster landing": [], "Date": [], "Time": []
    }

    tables = soup.find_all("table", class_="wikitable plainrowheaders collapsible")
    if not tables:
        tables = [
            t for t in soup.find_all("table")
            if "wikitable" in (t.get("class") or [])
            and "plainrowheaders" in (t.get("class") or [])
        ]

    for table in tables:
        for row_tag in table.find_all("tr"):
            if not row_tag.th or not row_tag.th.string:
                continue
            flight_no = row_tag.th.string.strip()
            if not flight_no.isdigit():
                continue

            row = row_tag.find_all("td")
            if len(row) < 9:
                continue

            launch_dict["Flight No."].append(flight_no)
            dt = date_time(row[0])
            launch_dict["Date"].append(dt[0].strip(","))
            launch_dict["Time"].append(dt[1])

            bv = booster_version(row[1])
            if not bv:
                bv = row[1].a.string if row[1].a is not None else row[1].get_text(strip=True)
            launch_dict["Version Booster"].append(bv)

            launch_dict["Launch site"].append(
                row[2].a.string if row[2].a is not None else row[2].get_text(strip=True)
            )
            launch_dict["Payload"].append(
                row[3].a.string if row[3].a is not None else row[3].get_text(strip=True)
            )
            launch_dict["Payload mass"].append(get_mass(row[4]))
            launch_dict["Orbit"].append(
                row[5].a.string if row[5].a is not None else row[5].get_text(strip=True)
            )

            try:
                customer = row[6].a.string or "Various"
            except Exception:
                customer = "Various"
            launch_dict["Customer"].append(customer)

            strings = list(row[7].strings)
            launch_dict["Launch outcome"].append(strings[0] if strings else "")
            launch_dict["Booster landing"].append(landing_status(row[8]))

    return pd.DataFrame(launch_dict)

## Execute the scraping workflow

If this runtime cannot reach Wikipedia, the notebook loads the repository's saved
result from the same fixed snapshot. This keeps the result reproducible without
inventing rows.

In [3]:
try:
    scraped = scrape_launch_table(WIKI_URL)
    if len(scraped) == 0:
        raise RuntimeError("No launch rows were parsed.")
    print("Live web scrape succeeded:", scraped.shape)
except Exception as exc:
    print("Live scrape unavailable in this runtime:")
    print(type(exc).__name__, "-", str(exc)[:180])
    scraped = pd.read_csv(SAVED_FILE)
    print("Loaded saved scraped result:", scraped.shape)

scraped.head()

Live scrape unavailable in this runtime:
ConnectionError - HTTPSConnectionPool(host='en.wikipedia.org', port=443): Max retries exceeded with url: /w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922 (Caused by Nam
Loaded saved scraped result: (121, 11)


,Flight No.,Launch site,Payload,Payload mass,Orbit,Customer,Launch outcome,Version Booster,Booster landing,Date,Time
0,1,CCAFS,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success\n,F9 v1.0B0003.1,Failure,4 June 2010,18:45
1,2,CCAFS,Dragon,0,LEO,NASA,Success,F9 v1.0B0004.1,Failure,8 December 2010,15:43
2,3,CCAFS,Dragon,525 kg,LEO,NASA,Success,F9 v1.0B0005.1,No attempt\n,22 May 2012,07:44
3,4,CCAFS,SpaceX CRS-1,"4,700 kg",LEO,NASA,Success\n,F9 v1.0B0006.1,No attempt,8 October 2012,00:35
4,5,CCAFS,SpaceX CRS-2,"4,877 kg",LEO,NASA,Success\n,F9 v1.0B0007.1,No attempt\n,1 March 2013,15:10


In [4]:
print("Rows:", len(scraped))
print("Columns:", list(scraped.columns))
print("\nLaunch outcomes:")
print(scraped["Launch outcome"].astype(str).str.strip().value_counts().head())

Rows: 121
Columns: ['Flight No.', 'Launch site', 'Payload', 'Payload mass', 'Orbit', 'Customer', 'Launch outcome', 'Version Booster', 'Booster landing', 'Date', 'Time']

Launch outcomes:
Launch outcome
Success    120
Failure      1
Name: count, dtype: int64
